# Battle Lab · ciclo completo M-C

Actualiza pastes y partidas humanas, entrena un candidato y lo compara con el modelo vigente. **GPU → Ejecutar todo**. El primer champion es el LIGHT M-C ya validado; requiere sus archivos existentes en Drive.

Cada ciclo guarda una copia fija de sus datos. Si se corta, `auto` reanuda la ejecución pendiente; cuando termina, la siguiente ejecución recoge novedades. La configuración debe coincidir para reanudar. `new` inicia otro experimento.

| Modo | Pasos PPO adicionales | Épocas BC, si hay datos suficientes |
|---|---:|---:|
| CENSUS | 0 | 0 |
| LIGHT | 196.608 | 3 |
| NORMAL | 786.432 | 5 |
| HARD | 3.145.728 | 10 |

BC necesita ≥1.000 trayectorias y ≥10.000 transiciones elegibles de ganadores con rating ≥1.200. Con menos datos se omite BC y se entrena mediante PPO/self-play. Estos umbrales son operativos, no una garantía de calidad.

Se conserva la separación histórica de equipos. Cada modelo juega contra los mismos tres controles y con el mismo calendario: por defecto **3.000 batallas** en total. El resultado mide mejora observada; no demuestra significancia estadística ni fuerza contra humanos.

Patrón operativo: **40_017 v3**, procesos nuevos, recursos auto90, coordinador único, progreso/ETA y checkpoints persistentes. Los resultados quedan en `Colabs/LikeNoOneEverWas/BattleLab/MC-Training/Refresh/`.


In [ ]:
#@title 1 · Configuración
RUN_MODE = "LIGHT" #@param ["CENSUS", "LIGHT", "NORMAL", "HARD"]
RUN_ACTION = "auto" #@param ["auto", "new", "resume"]
RUN_ID = "" #@param {type:"string"}
BATTLES_PER_CONTROL = 500 #@param {type:"integer"}
REPLAY_PAGES_PER_FORMAT = 100 #@param {type:"integer"}
DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
WORKERS = 0 #@param {type:"integer"}
NUM_ENVS = 2 #@param [1, 2, 4] {type:"raw"}
SEED = 260916 #@param {type:"integer"}
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "battle-lab-mc-refresh-001"
NODE_VERSION = "24.21.0"


In [ ]:
#@title 2 · Drive, ejecución con progreso y código verificable
from google.colab import drive, userdata
from pathlib import Path
import json, os, queue, re, signal, subprocess, sys, tempfile, threading, time

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
REFRESH = ROOT / "Refresh"
REPO = Path("/content/pkmn-mc-refresh")
RUNTIME = Path("/content/battle-lab-refresh-runtime")
(REFRESH / "extra-teams").mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, env=None):
    process = subprocess.Popen([str(x) for x in command], cwd=cwd, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    pending = queue.Queue()
    def pump():
        for line in process.stdout:
            pending.put(line)
    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    started = heartbeat = time.monotonic()
    try:
        while process.poll() is None or reader.is_alive() or not pending.empty():
            try:
                print(pending.get(timeout=.2), end="", flush=True)
            except queue.Empty:
                pass
            if time.monotonic() - heartbeat >= 15:
                print(f"⏳ Proceso activo · {int(time.monotonic()-started)} s", flush=True)
                heartbeat = time.monotonic()
        if process.returncode:
            raise RuntimeError(f"Proceso terminado con código {process.returncode}; revisa su salida.")
    finally:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.terminate()
        reader.join(timeout=2)

source_ref = PKMN_REF
active_path = REFRESH / "active_run.json"
pending_id = RUN_ID or (json.loads(active_path.read_text()).get("runId", "")
                         if RUN_ACTION != "new" and active_path.exists() else "")
if pending_id:
    if not re.fullmatch(r"[A-Za-z0-9_-]+", pending_id):
        raise ValueError("RUN_ID inválido")
    previous_dir = REFRESH / "runs" / pending_id
    status_path = previous_dir / "status.json"
    status = json.loads(status_path.read_text()) if status_path.exists() else {}
    if RUN_ID or status.get("state") not in ("completed", "census_completed"):
        source_ref = json.loads((previous_dir / "config.json").read_text())["codeSha"]

# El repositorio es público. Un secreto existente se usa solo en el entorno
# temporal de Git, nunca se inserta en una URL, un archivo persistente o la salida.
token = ""
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    pass
git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"
try:
    with tempfile.TemporaryDirectory() as private:
        if token:
            helper = Path(private) / "askpass.py"
            helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'Username' in sys.argv[1] else os.environ['PKMN_GITHUB_TOKEN'])\n")
            helper.chmod(0o700)
            git_env.update(GIT_ASKPASS=str(helper), PKMN_GITHUB_TOKEN=token)
        if not (REPO / ".git").is_dir():
            run(["git", "clone", PKMN_REPOSITORY, REPO], env=git_env)
        dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True)
        if dirty.strip():
            raise RuntimeError("El checkout temporal tiene cambios; revísalos antes de actualizarlo.")
        run(["git", "fetch", "origin", source_ref], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, env=git_env)
finally:
    token = ""
    git_env.pop("PKMN_GITHUB_TOKEN", None)
    git_env.pop("GIT_ASKPASS", None)
CODE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Código:", CODE_SHA)


In [ ]:
#@title 3 · Dependencias y comprobación previa
try:
    node_major = int(subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v").split(".")[0])
except (OSError, subprocess.CalledProcessError):
    node_major = 0
if node_major < 24:
    run(["npm", "install", "-g", "n"])
    run(["n", NODE_VERSION])
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
run([sys.executable, "-m", "pip", "install", "-q", "-r", "battle_lab/requirements-phase1.txt",
     "-r", "battle_lab/requirements-mc-train.txt"], cwd=REPO)
# Se usan procesos nuevos: una versión importada en el kernel nunca se reutiliza.
run([sys.executable, "-c", "from battle_lab.mc_refresh import PROFILES; from battle_lab.mc_refresh_data import resolve_workers; print('Preflight OK', PROFILES.keys(), resolve_workers())"], cwd=REPO)
run(["node", "--test", "tests/battle-lab-mc-refresh.test.mjs"], cwd=REPO)


## Ejecutar el ciclo

Puedes añadir pastes publicados completos en `Refresh/extra-teams/*.txt`. Se validan seis sets completos; no se inventan EVs, movimientos ni habilidades.

El scraper consulta primero las novedades y continúa el histórico pendiente dentro del presupuesto de páginas. Si se alcanza ese presupuesto, el informe lo distingue de una fuente agotada. Se excluyen las partidas que incluyen equipos reservados para evaluación.

La barra marca las fases completadas. Entrenamiento y evaluación muestran su propio progreso. El ETA por fase aparece cuando existe una ejecución anterior comparable; en la primera se está calculando.


In [ ]:
#@title 4 · Pastes → partidas → entrenamiento → prueba
command = [sys.executable, "-u", "-m", "battle_lab.mc_refresh", "--root", ROOT,
           "--runtime-root", RUNTIME, "--mode", RUN_MODE, "--run-action", RUN_ACTION,
           "--seed", SEED, "--workers", WORKERS, "--num-envs", NUM_ENVS,
           "--battles", BATTLES_PER_CONTROL, "--replay-pages", REPLAY_PAGES_PER_FORMAT,
           "--device", DEVICE]
if RUN_ID:
    command += ["--run-id", RUN_ID]
run(command, cwd=REPO)


In [ ]:
#@title 5 · Resultado y archivos de revisión
result = json.loads((REFRESH / "latest_result.json").read_text())
if result["state"] not in ("completed", "census_completed"):
    raise RuntimeError("No hay una ejecución completa para revisar.")
COMPLETED_RUN_ID = result["runId"]
COMPLETED_RUN = REFRESH / "runs" / COMPLETED_RUN_ID
report = (COMPLETED_RUN / "report.txt").read_text()
if not report.strip():
    raise RuntimeError("El informe de texto está vacío.")
print(report)
print("Para revisar con Roku/Gem: comparte Refresh/latest_run.txt o la carpeta", COMPLETED_RUN)
print("Detalles: report.json; comparación: comparison.csv; logs y replays: evaluation/.")


## Selección para el siguiente ciclo

Por defecto se conserva el champion. Después de revisar el informe, puedes activar esta celda para seleccionar un candidato con mejora observada. Esto cambia el punto de partida del **próximo ciclo**. La instalación del modelo en la ROG y Nana es un paso separado.

El grupo reservado se reutiliza para decisiones periódicas y acaba actuando como validación. El informe separa los equipos recién reservados; el gate heredado no sustituye una prueba final independiente ni una evaluación contra humanos. Tampoco se conoce todo el corpus previo del checkpoint público original.


In [ ]:
#@title 6 · Opcional: seleccionar candidato revisado
PROMOTE_CANDIDATE = False #@param {type:"boolean"}
if PROMOTE_CANDIDATE:
    run([sys.executable, "-m", "battle_lab.mc_refresh", "--root", ROOT,
         "--promote-run", COMPLETED_RUN_ID], cwd=REPO)
else:
    print("Champion conservado. El candidato y sus resultados quedan guardados.")
